In [2]:
from pathlib import Path
import h5py
import numpy as np


old_path = Path("data/3T/Vol01_WB/Res64x64_Thick/TrainData/TrainData_v_2.0_old_code.h5")
new_path = Path("data/3T/Vol01_WB/Res64x64_Thick/TrainData/TrainData_v_2.0_refactored.h5")

expected_equal = ["metab", "water", "lipid", "spectra"]
expected_different = ["lipid_proj", "lipid_projOP"]


def compare_dataset(name, old, new):
    print(f"\n[{name}]")
    print("old shape:", old.shape, old.dtype)
    print("new shape:", new.shape, new.dtype)
    print("same shape:", old.shape == new.shape)

    if old.shape != new.shape:
        return

    print("exact equal:", np.array_equal(old, new))
    print("allclose:", np.allclose(old, new))

    diff = old - new
    print("max abs diff:", np.max(np.abs(diff)))
    print("mean abs diff:", np.mean(np.abs(diff)))
    print("old finite:", np.all(np.isfinite(old)))
    print("new finite:", np.all(np.isfinite(new)))


with h5py.File(old_path, "r") as f_old, h5py.File(new_path, "r") as f_new:
    old_keys = sorted(list(f_old.keys()))
    new_keys = sorted(list(f_new.keys()))

    print("old keys:", old_keys)
    print("new keys:", new_keys)
    print("same keys:", old_keys == new_keys)

    print("\n=== Expected identical ===")
    for key in expected_equal:
        compare_dataset(key, f_old[key][:], f_new[key][:])

    print("\n=== Expected possibly different ===")
    for key in expected_different:
        compare_dataset(key, f_old[key][:], f_new[key][:])

old keys: ['lipid', 'lipid_proj', 'lipid_projOP', 'metab', 'spectra', 'water']
new keys: ['lipid', 'lipid_proj', 'lipid_projOP', 'metab', 'spectra', 'water']
same keys: True

=== Expected identical ===

[metab]
old shape: (10, 288) complex128
new shape: (10, 288) complex128
same shape: True
exact equal: True
allclose: True
max abs diff: 0.0
mean abs diff: 0.0
old finite: True
new finite: True

[water]
old shape: (10, 288) complex128
new shape: (10, 288) complex128
same shape: True
exact equal: True
allclose: True
max abs diff: 0.0
mean abs diff: 0.0
old finite: True
new finite: True

[lipid]
old shape: (10, 288) complex128
new shape: (10, 288) complex128
same shape: True
exact equal: True
allclose: True
max abs diff: 0.0
mean abs diff: 0.0
old finite: True
new finite: True

[spectra]
old shape: (10, 288) complex128
new shape: (10, 288) complex128
same shape: True
exact equal: True
allclose: True
max abs diff: 0.0
mean abs diff: 0.0
old finite: True
new finite: True

=== Expected possib

In [3]:
def rel_stats(old, new, eps=1e-12):
    diff = new - old

    rel_to_old = np.abs(diff) / (np.abs(old) + eps)
    rel_to_scale = np.linalg.norm(diff.ravel()) / (np.linalg.norm(old.ravel()) + eps)

    print("mean abs diff:", np.mean(np.abs(diff)))
    print("max abs diff:", np.max(np.abs(diff)))
    print("mean relative elementwise diff:", np.mean(rel_to_old))
    print("median relative elementwise diff:", np.median(rel_to_old))
    print("max relative elementwise diff:", np.max(rel_to_old))
    print("relative Frobenius/L2 diff:", rel_to_scale)


with h5py.File(old_path, "r") as f_old, h5py.File(new_path, "r") as f_new:
    print("\n=== lipid_projOP ===")
    old_op = f_old["lipid_projOP"][:]
    new_op = f_new["lipid_projOP"][:]
    rel_stats(old_op, new_op)

    print("\n=== lipid_proj ===")
    old_proj = f_old["lipid_proj"][:]
    new_proj = f_new["lipid_proj"][:]
    rel_stats(old_proj, new_proj)


=== lipid_projOP ===
mean abs diff: 0.00017302141143521756
max abs diff: 0.016708121225710578
mean relative elementwise diff: 0.02272120818497111
median relative elementwise diff: 0.01760032786085781
max relative elementwise diff: 2.7319576477787413
relative Frobenius/L2 diff: 0.03413838265530727

=== lipid_proj ===
mean abs diff: 112.03274328374583
max abs diff: 2522.0439314426876
mean relative elementwise diff: 0.014686820365963213
median relative elementwise diff: 0.00959354493399404
max relative elementwise diff: 0.3264426394229836
relative Frobenius/L2 diff: 0.014444327744294543
